In [4]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import category_encoders as ce
import shap


# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading and preparing data")
print("=" * 60)

df_raw = pd.read_excel(r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\microrna_clean.xlsx')

# ── Build miRNA lookup BEFORE dropping the microrna column ────
# Keys  : exact miRBase IDs as they appear in the data (e.g. 'hsa-miR-146b')
# Values: dict with pct_up, n, always_up, always_down
def build_mirna_lookup(df_source):
    df_source = df_source.copy()
    df_source['seed_family'] = df_source['seed_family'].replace('not_broadly_conserved', np.nan)
    lookup = {}
    for mirna, grp in df_source.groupby('microrna'):
        n      = len(grp)
        n_up   = grp['is_upregulated'].sum()
        pct_up = n_up / n
        lookup[mirna] = {
            'pct_up':      round(pct_up, 3),
            'n':           n,
            'always_up':   pct_up == 1.0,
            'always_down': pct_up == 0.0,
            'mixed':       0 < pct_up < 1.0,
            'seed_family': grp['seed_family'].iloc[0],
        }
    return lookup

mirna_lookup = build_mirna_lookup(df_raw)
print(f"miRNA lookup built: {len(mirna_lookup)} unique miRNAs")

# ── Also build seed family lookup for unseen miRNAs ───────────
def build_family_lookup(df_source):
    df_source = df_source.copy()
    df_source['seed_family'] = df_source['seed_family'].replace('not_broadly_conserved', np.nan)
    lookup = {}
    for fam, grp in df_source.dropna(subset=['seed_family']).groupby('seed_family'):
        n      = len(grp)
        pct_up = grp['is_upregulated'].mean()
        lookup[fam] = {
            'pct_up':      round(pct_up, 3),
            'n':           n,
            'always_up':   pct_up == 1.0,
            'always_down': pct_up == 0.0,
            'mixed':       0 < pct_up < 1.0,
        }
    return lookup

family_lookup = build_family_lookup(df_raw)
print(f"Family lookup built: {len(family_lookup)} seed families")

# ── Now proceed with original preprocessing ───────────────────
df = df_raw.copy()
df['parasite'] = df['parasite'].str.replace(r'\s+', '', regex=True)
df = df.drop(columns=['microrna', 'infection', 'microrna_group_simplified'])

# Non-conserved / still missing → NaN
df['seed_family'] = df['seed_family'].replace('not_broadly_conserved', np.nan)

# is_conserved flag: 1 if seed family is known, 0 if not
df['is_conserved'] = df['seed_family'].notna().astype(int)

# Interaction feature: parasite × cell type
df['parasite_celltype'] = df['parasite'].astype(str) + '_' + df['cell type'].astype(str)

print(f"Total rows            : {len(df)}")
print(f"Seed family known     : {df['is_conserved'].sum()}")
print(f"Seed family unknown   : {(df['is_conserved']==0).sum()}")
print(f"Target balance        :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Features and target
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Features and target")
print("=" * 60)

TARGET   = 'is_upregulated'
CAT_COLS = ['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']
NUM_COLS = ['time', 'is_conserved']

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Categorical   : {CAT_COLS}")
print(f"Numeric       : {NUM_COLS}")
print(f"Total features: {len(CAT_COLS) + len(NUM_COLS)}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Cross-Validation with Optuna best parameters
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Cross-Validation (5-fold, Optuna best params)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BEST_PARAMS = {
    'n_estimators':      303,
    'max_depth':         7,
    'learning_rate':     0.00604,
    'num_leaves':        24,
    'min_child_samples': 6,
    'subsample':         0.9999,
    'colsample_bytree':  0.7345,
    'reg_alpha':         0.01531,
    'reg_lambda':        5.21e-6,
    'random_state':      42,
    'verbose':           -1,
    'class_weight':      'balanced',
}

pipe = Pipeline([
    ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
    ('model',   LGBMClassifier(**BEST_PARAMS))
])

auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
f1  = cross_val_score(pipe, X, y, cv=cv, scoring='f1')

print(f"ROC-AUC : {auc.mean():.3f} ± {auc.std():.3f}")
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"F1      : {f1.mean():.3f} ± {f1.std():.3f}")
print(f"AUC per fold: {[round(x, 3) for x in auc.tolist()]}")


# ─────────────────────────────────────────────────────────────
# STEP 4 — Train final model on all data
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Training final model on all data")
print("=" * 60)

pipe.fit(X, y)
print("Done.")


# ─────────────────────────────────────────────────────────────
# STEP 5 — Permutation Feature Importance (leak-free)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Permutation Feature Importance (leak-free)")
print("=" * 60)

fold_importances = []

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    fold_pipe = Pipeline([
        ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
        ('model',   LGBMClassifier(**BEST_PARAMS))
    ])
    fold_pipe.fit(X_train, y_train)

    X_val_enc = fold_pipe.named_steps['encoder'].transform(X_val)
    perm = permutation_importance(
        fold_pipe.named_steps['model'],
        X_val_enc, y_val,
        n_repeats=10, random_state=42, scoring='roc_auc'
    )
    fold_importances.append(perm.importances_mean)
    print(f"  Fold {fold_idx + 1} done.")

mean_importances = np.mean(fold_importances, axis=0)
std_importances  = np.std(fold_importances,  axis=0)

perm_df = pd.DataFrame({
    'feature':    X.columns.tolist(),
    'importance': mean_importances,
    'std':        std_importances
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>10} {'Std':>8}")
print("-" * 47)
for _, row in perm_df.iterrows():
    bar  = '█' * max(0, int(row['importance'] * 60))
    sign = '+' if row['importance'] >= 0 else ''
    print(f"  {row['feature']:<23} {sign}{row['importance']:.4f} ± {row['std']:.4f}  {bar}")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Build SHAP explainer
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Building SHAP explainer")
print("=" * 60)

X_enc = pipe.named_steps['encoder'].transform(X)
explainer = shap.TreeExplainer(pipe.named_steps['model'])
shap_vals = explainer.shap_values(X_enc)
print(f"SHAP values shape : {np.array(shap_vals).shape}  ✓")


# ─────────────────────────────────────────────────────────────
# STEP 7 — Save model bundle
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Saving model bundle")
print("=" * 60)

model_bundle = {
    'model':     pipe,
    'encoder':   pipe.named_steps['encoder'],
    'lgbm':      pipe.named_steps['model'],
    'explainer': explainer,
    'metrics': {
        'auc_mean':   round(auc.mean(), 3),
        'auc_std':    round(auc.std(),  3),
        'acc_mean':   round(acc.mean(), 3),
        'acc_std':    round(acc.std(),  3),
        'f1_mean':    round(f1.mean(),  3),
        'f1_std':     round(f1.std(),   3),
        'auc_folds':  [round(x, 3) for x in auc.tolist()],
        'best_params': BEST_PARAMS,
        'n_train':    len(X),
        'feature_importance': perm_df.to_dict('records'),
    },
    'options': {
        'parasite':      sorted(df['parasite'].dropna().unique().tolist()),
        'organism':      sorted(df['organism'].dropna().unique().tolist()),
        'cell_type':     sorted(df['cell type'].dropna().unique().tolist()),
        'time':          sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df['seed_family'].dropna().unique().tolist()),
    },
    'cat_cols':      CAT_COLS,
    'feature_names': list(X.columns),
    # ── Lookup dicts for Streamlit app ────────────────────────
    # Used at prediction time only — not features, not in the model
    'mirna_lookup':  mirna_lookup,   # miRNA name  → {pct_up, n, always_up, always_down, mixed, seed_family}
    'family_lookup': family_lookup,  # seed family → {pct_up, n, always_up, always_down, mixed}
}

with open(r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\lgbm_mirna_model_fixed.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print("Saved: lgbm_mirna_model.pkl")

STEP 1 — Loading and preparing data
miRNA lookup built: 115 unique miRNAs
Family lookup built: 40 seed families
Total rows            : 206
Seed family known     : 138
Seed family unknown   : 68
Target balance        :
is_upregulated
1    103
0    103

STEP 2 — Features and target
Categorical   : ['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']
Numeric       : ['time', 'is_conserved']
Total features: 7

STEP 3 — Cross-Validation (5-fold, Optuna best params)
ROC-AUC : 0.866 ± 0.061
Accuracy: 0.801 ± 0.061
F1      : 0.789 ± 0.070
AUC per fold: [0.837, 0.9, 0.963, 0.783, 0.845]

STEP 4 — Training final model on all data
Done.

STEP 5 — Permutation Feature Importance (leak-free)
  Fold 1 done.
  Fold 2 done.
  Fold 3 done.
  Fold 4 done.
  Fold 5 done.

Feature                   Importance      Std
-----------------------------------------------
  seed_family             +0.2128 ± 0.0521  ████████████
  time                    +0.1483 ± 0.0722  ████████
  parasite_